# NCA K5 | Seasonal-median XGBoost retrieval (2016–2026)

**Standalone supervised-learning notebook — seasonal Sentinel-2 medians only.** No harmonic predictors are used.

- Join contemporaneous **field_year = spectral_year** median tables to the original field master and separate 2026 vegetation file; retain all unmatched field observations in the exported master for provenance.
- For **2016**, exclude site records that have **no valid monthly observations at all** from the ML-eligible sample. This is a *data-coverage filter*, not an exact historical tile-footprint mask: an unobserved pixel might also be due to clouds or masking.
- Derive spring (Mar–May), summer (Jun–Aug), autumn (Sep–Nov) means of valid monthly medians (15 variables × 3 seasons = **45 spectral predictors**) plus Year/Easting/Northing = **48 predictors**.
- Fit three familiar supervised targets: seven two-part fractional-cover models, five-dimensional K5 membership (four additive log-ratio regressions), and direct five-class dominant-state classification.
- Hold out entire **coordinate-defined sites** rather than splitting repeated field observations across partitions. A single grouped split is an exploratory benchmark, *not* independent-year or spatial-block validation.

**Working directory:** place this notebook in the same directory as the observatory notebook. Source CSVs should be under `data/monthly_medians/`; outputs are written under `outputs/monthly_medians/seasonal_xgboost/`.


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             accuracy_score, balanced_accuracy_score,
                             classification_report, ConfusionMatrixDisplay)

SEED = 622
DATA_DIR = Path('data') / 'monthly_medians'
OUTPUT_DIR = Path('outputs') / 'monthly_medians' / 'seasonal_xgboost'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MASTER = DATA_DIR / 'NCA_K5_PlotYear_Master.csv'
VEG2026 = DATA_DIR / 'NCA_K5_2026_Vegetation_Spatial.csv'
MEDIAN_GLOB = 'NCA_S2_monthly_medians_sites_*.csv'

FG_COLS = ['EAG_FG','EF_FG','PBG_FG','SHRUB_FG',
           'BAREGROUND_FG','LITTER_FG','BIOCRUST_FG']
FG_LABELS = ['EAG','EF','PBG','SHRUB','Bare ground','Litter','Biocrust']
K5_COLS = [f'membership_k5_{i}' for i in range(1,6)]
K5_LABELS = ['PBG','ARTR','SHRUB','EF','EAG']
SPECTRAL_VARS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12',
                 'NDVI','NDMI','NDRE','BSI','NBR2']
MONTHS = list(range(1,13))
SEASONS = {'spring':[3,4,5], 'summer':[6,7,8], 'autumn':[9,10,11]}

# Exploratory default: retain seasons with at least one observed month.
# Tighten to 2 / 2 for a sensitivity test after the first run.
MIN_MONTH_NOBS = 1
MIN_VALID_MONTHS = 1
SITE_GROUP_ROUND_M = 1.0  # repeated observations within a 1m grid cell held in same split
MAX_JOIN_DISTANCE_M = 20.0  # reject apparent PlotID/year misjoins
TEST_SIZE = 0.20
XGB_PARAMS = dict(n_estimators=350, max_depth=3, learning_rate=0.04,
                  subsample=0.8, colsample_bytree=0.85,
                  reg_lambda=5.0, min_child_weight=5,
                  tree_method='hist', n_jobs=4, random_state=SEED)

MONTHLY_COLS = [f'm{m:02d}_{v}' for m in MONTHS for v in SPECTRAL_VARS]
COUNT_COLS = [f'm{m:02d}_nObs' for m in MONTHS]
SEASON_COLS = [f'{s}_{v}' for s in SEASONS for v in SPECTRAL_VARS]
FEATURES = SEASON_COLS + ['Year','Easting','Northing']
assert len(FEATURES) == 48
print('Source directory:', DATA_DIR.resolve())
print('xgboost:', xgb.__version__)


Source directory: C:\Users\scottfordham\BOP_OCTC\Python\BOP Datapull\Medians\data\monthly_medians
xgboost: 3.4.1


## 1. Load the two vegetation sources

The 2026 table is authoritative for its year. Existing 2026 records are excluded from the older master before combining, preventing duplicate 2026 field observations. Cover stays in percent in the saved master and is converted to fractions **only in the modeling dataframe**.


In [ ]:
# ============================================================
# LOAD TWO VEGETATION SOURCES
# Standardize plot identifiers before combining
# ============================================================

master = pd.read_csv(
    DATA_DIR / "NCA_K5_PlotYear_Master.csv",
    low_memory=False
)

veg2026 = pd.read_csv(
    DATA_DIR / "NCA_K5_2026_Vegetation_Spatial.csv",
    low_memory=False
)

# Original master: PlotID
# 2026 vegetation: Plot
veg2026 = veg2026.rename(columns={
    "Plot": "PlotID"
})

# Standardize identifier types.
for df in [master, veg2026]:
    df["PlotID"] = (
        df["PlotID"]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    df["Year"] = pd.to_numeric(
        df["Year"], errors="raise"
    ).astype(int)

# Preserve the 2026 file as the authoritative source for 2026.
master = master.loc[master["Year"] != 2026].copy()

master["vegetation_source"] = "existing_master"
veg2026["vegetation_source"] = "2026_vegetation"

vegetation = pd.concat(
    [master, veg2026],
    ignore_index=True,
    sort=False
)

print(
    vegetation.groupby(
        ["Year", "vegetation_source"]
    ).size()
)


# ============================================================
# RESTORE VEGETATION JOIN KEYS
# Required by downstream median-matching cells
# ============================================================

vegetation["join_id"] = (
    vegetation["PlotID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

vegetation["join_year"] = pd.to_numeric(
    vegetation["Year"],
    errors="raise"
).astype(int)

# Verify required fields are available.
required = [
    "join_id",
    "join_year",
    "Easting",
    "Northing"
]

missing = [
    col for col in required
    if col not in vegetation.columns
]

if missing:
    raise KeyError(
        f"Vegetation master missing required fields: {missing}"
    )

print(
    f"{len(vegetation):,} vegetation observations "
    "prepared for spatial matching."
)


Year  vegetation_source
2016  existing_master      255
2018  existing_master      106
2025  existing_master      118
2026  2026_vegetation      454
dtype: int64


## 2. Load annual spectral tables and match field-observation years

Every annual satellite CSV contains all site locations. **Only rows with `field_year == spectral_year` are eligible as field labels.** Identical duplicated exports are collapsed; where a PlotID/year produces multiple possible points, the closest coordinate match to the vegetation source is used, subject to a 20 m maximum separation.


In [ ]:
files = sorted(DATA_DIR.glob(MEDIAN_GLOB))
if not files:
    raise FileNotFoundError(f'No {MEDIAN_GLOB} files under {DATA_DIR}')
parts = []
metadata = ['plot_id','field_year','spectral_year','Easting','Northing']
expected = metadata + MONTHLY_COLS + COUNT_COLS
for path in files:
    f = pd.read_csv(path, dtype={'plot_id':'string'}, low_memory=False)
    missing = set(expected).difference(f.columns)
    if missing:
        raise ValueError(f'{path.name}: missing {len(missing)} columns (e.g. {sorted(missing)[:8]})')
    f = f[expected].copy()
    f['source_file'] = path.name
    parts.append(f)
median_all = pd.concat(parts, ignore_index=True)
for c in ['field_year','spectral_year','Easting','Northing',*MONTHLY_COLS,*COUNT_COLS]:
    median_all[c] = pd.to_numeric(median_all[c], errors='coerce')
median_all[MONTHLY_COLS] = median_all[MONTHLY_COLS].replace(-32768, np.nan)
median_all['join_id'] = normalize_plot_id(median_all['plot_id'])
median_all['join_year'] = median_all['field_year']
median_field = median_all.loc[
    median_all['field_year'].eq(median_all['spectral_year']) &
    median_all['join_id'].notna() & median_all['join_year'].notna()
].copy()

# Exact duplicate rows do not add independent information.
median_field = median_field.drop_duplicates(
    subset=['join_id','join_year','Easting','Northing',*MONTHLY_COLS,*COUNT_COLS]
)

# Allow candidate duplicates in median exports, then use coordinates to choose
# a uniquely closest matching record per vegetation observation.
_candidates = vegetation[['join_id','join_year','Easting','Northing']].merge(
    median_field, on=['join_id','join_year'], how='left',
    suffixes=('_veg','_median'), indicator=True
)
_candidates['join_distance_m'] = np.hypot(
    _candidates['Easting_veg'] - _candidates['Easting_median'],
    _candidates['Northing_veg'] - _candidates['Northing_median']
)
_candidates = _candidates.sort_values(['join_id','join_year','join_distance_m'],
                                      na_position='last')
# One nearest candidate per original field observation identity.
matched_med = _candidates.drop_duplicates(['join_id','join_year']).copy()
bad_distance = matched_med['join_distance_m'].gt(MAX_JOIN_DISTANCE_M)
if bad_distance.any():
    raise ValueError(f'{bad_distance.sum()} plot-year matches > {MAX_JOIN_DISTANCE_M} m apart; audit IDs/CRS')
matched_med['has_median_export'] = matched_med['_merge'].eq('both')

keep = ['join_id','join_year','has_median_export','join_distance_m',
        *MONTHLY_COLS,*COUNT_COLS]
median_master = vegetation.merge(matched_med[keep], on=['join_id','join_year'],
                                 how='left', validate='one_to_one')
median_master['has_median_export'] = median_master['has_median_export'].fillna(False).astype(bool)
median_master['total_valid_daily_observations'] = (
    median_master[COUNT_COLS].fillna(0).sum(axis=1)
)
# For 2016 the historical image asset covers only certain tiles. A row with
# no usable observations is excluded from supervised learning, but retained
# in the saved field master. This is NOT an exact tile-footprint determination.
median_master['spectral_coverage_available'] = (
    median_master['total_valid_daily_observations'] > 0
)
print('Annual CSVs:', [p.name for p in files])
display(median_master.groupby('Year').agg(
    field_n=('PlotID','size'),
    matched=('has_median_export','sum'),
    any_valid_spectral=('spectral_coverage_available','sum')
))


C:\Users\scottfordham\AppData\Local\Temp\ipykernel_30416\92051469.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  median_all['join_id'] = normalize_plot_id(median_all['plot_id'])
C:\Users\scottfordham\AppData\Local\Temp\ipykernel_30416\92051469.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  median_all['join_year'] = median_all['field_year']


KeyError: "['join_id', 'join_year'] not in index"

## 3. Seasonal features and the definitive median–vegetation master

Seasonal features are **equal-weight means of eligible monthly medians**, not medians of all daily observations across the season. Missing values are never replaced by harmonic estimates. The 2016 historical-tile limitation is handled through the availability flags; rows without valid historical observations are retained in the master but omitted from model training.


In [ ]:
for season, months in SEASONS.items():
    sufficient_months = []
    for m in months:
        sufficient_months.append(
            median_master[f'm{m:02d}_nObs'].ge(MIN_MONTH_NOBS)
        )
    median_master[f'{season}_n_valid_months'] = pd.concat(sufficient_months, axis=1).sum(axis=1)
    for v in SPECTRAL_VARS:
        values = pd.concat([
            median_master[f'm{m:02d}_{v}'].where(
                median_master[f'm{m:02d}_nObs'].ge(MIN_MONTH_NOBS)
            ) for m in months
        ], axis=1)
        median_master[f'{season}_{v}'] = values.mean(axis=1).where(
            values.notna().sum(axis=1).ge(MIN_VALID_MONTHS)
        )

cover = median_master[FG_COLS].to_numpy(float)
U = median_master[K5_COLS].to_numpy(float)
median_master['valid_cover'] = (
    np.isfinite(cover).all(axis=1) &
    (cover >= 0).all(axis=1) & (cover <= 100).all(axis=1)
)
median_master['valid_membership'] = (
    np.isfinite(U).all(axis=1) & (U > 0).all(axis=1) &
    (U < 1).all(axis=1) & np.isclose(np.nansum(U, axis=1), 1, atol=1e-5)
)
median_master['complete_seasonal48'] = median_master[FEATURES].notna().all(axis=1)
median_master['eligible_ml'] = (
    median_master['valid_cover'] & median_master['valid_membership'] &
    median_master['complete_seasonal48'] & median_master['spectral_coverage_available']
)
median_master.to_csv(OUTPUT_DIR / 'NCA_K5_MonthlyMedians_Master.csv', index=False)

print('Joined master saved:', OUTPUT_DIR / 'NCA_K5_MonthlyMedians_Master.csv')
display(median_master.groupby('Year').agg(
    field_n=('PlotID','size'),
    any_spectral=('spectral_coverage_available','sum'),
    seasonal_complete=('complete_seasonal48','sum'),
    eligible_ml=('eligible_ml','sum')
))

model_df = median_master.loc[median_master['eligible_ml']].copy()
model_df[FG_COLS] = model_df[FG_COLS] / 100.0
if len(model_df) < 20:
    raise ValueError('Too few eligible observations; inspect seasonal coverage and joins')

# Hold repeated field visits at nearly identical coordinates together.
model_df['site_group'] = (
    (model_df['Easting'] / SITE_GROUP_ROUND_M).round().astype('int64').astype(str)
    + '::' +
    (model_df['Northing'] / SITE_GROUP_ROUND_M).round().astype('int64').astype(str)
)
print('Model observations:', len(model_df), '| distinct coordinate groups:',
      model_df['site_group'].nunique())


## 4. Grouped held-out evaluation

Keep all observations of a coordinate-defined site in the same partition. The split is held constant across the three response tasks. If the data combine field years sampled in different geographic regions, site grouping **does not** constitute spatial-block validation.


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(model_df, groups=model_df['site_group']))
train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()
assert set(train['site_group']).isdisjoint(set(test['site_group']))
Xtr = train[FEATURES]
Xte = test[FEATURES]
train_keys = train[['PlotID','Year','site_group']].copy()
train_keys['partition'] = 'train'
test_keys = test[['PlotID','Year','site_group']].copy()
test_keys['partition'] = 'test'
pd.concat([train_keys,test_keys]).to_csv(OUTPUT_DIR / 'site_grouped_split.csv', index=False)
print('Train:', len(train), 'plot-years | Test:', len(test), 'plot-years')
print('Train years:', train['Year'].value_counts().sort_index().to_dict())
print('Test years: ', test['Year'].value_counts().sort_index().to_dict())


## 5. Seven fractional-cover responses — two-part XGBoost

Fit `Pr(cover > 0 | X)` and `E(cover | cover > 0, X)` separately; combine their predictions. This is a **two-part/hurdle-style point predictor**, not a Bayesian zero-inflated beta likelihood. Cover is trained on the 0–1 fraction scale.


In [ ]:
fg_predictions = pd.DataFrame({'PlotID':test['PlotID'].to_numpy(),
                               'Year':test['Year'].to_numpy()})
fg_metrics_rows = []
fg_models = {}
for col, label in zip(FG_COLS, FG_LABELS):
    ytr = train[col].to_numpy(float)
    yte = test[col].to_numpy(float)
    presence = (ytr > 0).astype(int)
    positive = ytr > 0
    clf = None
    reg = None
    if np.unique(presence).size == 2:
        clf = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', **XGB_PARAMS)
        clf.fit(Xtr, presence)
        p = clf.predict_proba(Xte)[:,1]
    else:
        p = np.full(len(test), presence.mean())
    if positive.sum() >= 2 and np.unique(ytr[positive]).size > 1:
        reg = xgb.XGBRegressor(objective='reg:squarederror', **XGB_PARAMS)
        reg.fit(Xtr.loc[positive], ytr[positive])
        mu = np.clip(reg.predict(Xte), 0, 1)
    else:
        mu = np.full(len(test), ytr[positive].mean() if positive.any() else 0.)
    pred = p * mu
    fg_models[col] = {'occurrence':clf,'positive':reg}
    fg_predictions[f'{col}_observed'] = yte
    fg_predictions[f'{col}_predicted'] = pred
    fg_metrics_rows.append(dict(group=label, n_positive_train=int(positive.sum()),
                               MAE=mean_absolute_error(yte,pred),
                               RMSE=np.sqrt(mean_squared_error(yte,pred)),
                               Bias=np.mean(pred-yte),
                               Baseline_MAE=mean_absolute_error(yte,np.repeat(ytr.mean(),len(yte)))))
fg_metrics = pd.DataFrame(fg_metrics_rows)
fg_metrics.to_csv(OUTPUT_DIR / 'cover_heldout_metrics.csv', index=False)
fg_predictions.to_csv(OUTPUT_DIR / 'cover_heldout_predictions.csv', index=False)
display(fg_metrics.round(4))


## 6. Five-dimensional fuzzy K5 memberships — additive log ratios

Model the composition `membership_k5_1...5`, **not** observed vegetation fractions. Convert four predicted log ratios back to a strictly positive, sum-to-one vector; compare held-out predictions using Hellinger distance, dominant-state agreement, and normalized entropy error.


In [ ]:
def hellinger(a,b):
    return np.sqrt(((np.sqrt(a)-np.sqrt(b))**2).sum(axis=1)/2)

def normalized_entropy(u):
    return -(u*np.log(u)).sum(axis=1)/np.log(u.shape[1])

Utr = train[K5_COLS].to_numpy(float)
Ute = test[K5_COLS].to_numpy(float)
Ztr = np.log(Utr[:,1:] / Utr[:,[0]])
Zpred = np.empty((len(test),4), dtype=float)
k5_models = []
for k in range(4):
    model = xgb.XGBRegressor(objective='reg:squarederror', **XGB_PARAMS)
    model.fit(Xtr,Ztr[:,k])
    Zpred[:,k] = model.predict(Xte)
    k5_models.append(model)
logits = np.column_stack([np.zeros(len(test)), Zpred])
logits -= logits.max(axis=1, keepdims=True)
ex = np.exp(logits)
Upred = ex / ex.sum(axis=1, keepdims=True)

membership_metrics = pd.DataFrame([dict(
    n_test=len(test),
    mean_hellinger=hellinger(Ute,Upred).mean(),
    dominant_state_accuracy=(Ute.argmax(axis=1)==Upred.argmax(axis=1)).mean(),
    entropy_MAE=np.abs(normalized_entropy(Ute)-normalized_entropy(Upred)).mean(),
    observed_endmember_rate=np.mean(Ute.max(axis=1)>0.8),
    predicted_endmember_rate=np.mean(Upred.max(axis=1)>0.8)
)])
per_state_membership = pd.DataFrame([
    {'state':K5_LABELS[k],
     'MAE':mean_absolute_error(Ute[:,k],Upred[:,k]),
     'RMSE':np.sqrt(mean_squared_error(Ute[:,k],Upred[:,k]))}
    for k in range(5)
])
member_predictions = test[['PlotID','Year']].reset_index(drop=True).copy()
for k,c in enumerate(K5_COLS):
    member_predictions[f'{c}_observed']=Ute[:,k]
    member_predictions[f'{c}_predicted']=Upred[:,k]
member_predictions['hellinger']=hellinger(Ute,Upred)
member_predictions['dominant_observed']=np.array(K5_LABELS)[Ute.argmax(axis=1)]
member_predictions['dominant_predicted']=np.array(K5_LABELS)[Upred.argmax(axis=1)]
member_predictions.to_csv(OUTPUT_DIR/'membership_heldout_predictions.csv',index=False)
membership_metrics.to_csv(OUTPUT_DIR/'membership_heldout_summary.csv',index=False)
per_state_membership.to_csv(OUTPUT_DIR/'membership_heldout_by_state.csv',index=False)
display(membership_metrics.round(4))
display(per_state_membership.round(4))


## 7. Direct five-class dominant-state retrieval

The target is the **argmax of observed fuzzy K5 memberships**. Compare the direct classifier with the class implied by predicted memberships. No cover response is substituted for K5 state identity.


In [ ]:
ytr = Utr.argmax(axis=1)
yte = Ute.argmax(axis=1)
state_classifier = xgb.XGBClassifier(
    objective='multi:softprob', num_class=5,
    eval_metric='mlogloss', **XGB_PARAMS
)
state_classifier.fit(Xtr,ytr)
state_pred = state_classifier.predict(Xte).astype(int)
state_prob = state_classifier.predict_proba(Xte)

state_results = pd.DataFrame([{
    'direct_accuracy':accuracy_score(yte,state_pred),
    'direct_balanced_accuracy':balanced_accuracy_score(yte,state_pred),
    'membership_argmax_accuracy':accuracy_score(yte,Upred.argmax(axis=1)),
    'majority_train_baseline':np.mean(yte==np.bincount(ytr,minlength=5).argmax()),
    'n_test':len(yte)
}])
display(state_results.round(4))
print(classification_report(yte,state_pred,labels=np.arange(5),
                            target_names=K5_LABELS,zero_division=0))

state_out=test[['PlotID','Year']].reset_index(drop=True).copy()
state_out['observed_state']=np.array(K5_LABELS)[yte]
state_out['predicted_state']=np.array(K5_LABELS)[state_pred]
state_out['max_observed_membership']=Ute.max(axis=1)
for k, label in enumerate(K5_LABELS):
    state_out[f'p_{label}'] = state_prob[:,k]
state_out.to_csv(OUTPUT_DIR/'direct_state_heldout_predictions.csv',index=False)
state_results.to_csv(OUTPUT_DIR/'direct_state_heldout_summary.csv',index=False)
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(yte,state_pred,labels=np.arange(5),
                                        display_labels=K5_LABELS,cmap='Blues',ax=ax)
ax.set(title='Seasonal-median XGBoost | held-out dominant state',
       xlabel='Predicted',ylabel='Observed')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'dominant_state_confusion.png',dpi=200)
plt.show()


## 8. Compact visual diagnostics

These are held-out predictions only. An observed–predicted plot cannot distinguish spatial support mismatch from model error; it can identify which cover groups and membership components warrant further investigation.


In [ ]:
fig,axs=plt.subplots(4,2,figsize=(9,13),constrained_layout=True)
for ax,col,label in zip(axs.flat,FG_COLS,FG_LABELS):
    ax.scatter(fg_predictions[f'{col}_predicted'],fg_predictions[f'{col}_observed'],
               s=12,alpha=.55)
    ax.plot([0,1],[0,1],color='0.6',lw=.8)
    ax.set(xlim=(0,1),ylim=(0,1),title=label,
           xlabel='Predicted cover',ylabel='Observed cover')
    ax.set_aspect('equal',adjustable='box')
axs.flat[-1].axis('off')
plt.savefig(OUTPUT_DIR/'cover_heldout_scatter.png',dpi=200)
plt.show()

fig,axs=plt.subplots(3,2,figsize=(9,11),constrained_layout=True)
for k,(ax,label) in enumerate(zip(axs.flat,K5_LABELS)):
    ax.scatter(Upred[:,k],Ute[:,k],s=12,alpha=.55)
    ax.plot([0,1],[0,1],color='0.6',lw=.8)
    ax.set(xlim=(0,1),ylim=(0,1),title=label,
           xlabel='Predicted membership',ylabel='Observed membership')
    ax.set_aspect('equal',adjustable='box')
axs.flat[-1].axis('off')
plt.savefig(OUTPUT_DIR/'membership_heldout_scatter.png',dpi=200)
plt.show()


## 9. Seasonal feature importance (descriptive)

Mean XGBoost split importance is displayed for the four log-ratio models and the direct state classifier. Strongly correlated months/seasons and bands can substitute for one another; these values are **not** causal or unique contributions.


In [ ]:
importance = pd.DataFrame({
    'feature':FEATURES,
    'direct_state_importance':state_classifier.feature_importances_,
    'mean_membership_importance':np.mean(
        np.vstack([m.feature_importances_ for m in k5_models]),axis=0)
})
importance.to_csv(OUTPUT_DIR/'seasonal_feature_importance.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,col,title in [
    (axs[0],'direct_state_importance','Dominant state'),
    (axs[1],'mean_membership_importance','Membership log ratios')
]:
    a=importance.nlargest(12,col).sort_values(col)
    ax.barh(a['feature'],a[col])
    ax.set(xlabel='XGBoost feature importance',title=title)
plt.savefig(OUTPUT_DIR/'seasonal_feature_importance.png',dpi=200)
plt.show()


## Interpretation and next checks

- **2016 coverage:** the availability filter removes sites with no valid monthly measurements; it does **not** separately identify whether a point falls outside the historical scene footprints versus inside a masked pixel. An explicit historical-footprint polygon would distinguish these.
- **Seasonal completeness:** default is at least one month with `nObs ≥ 1` in each season. Increase `MIN_MONTH_NOBS` and `MIN_VALID_MONTHS` to test sensitivity to temporal support; do **not** compare scores across changed eligible samples as if only model quality changed.
- **2026:** autumn (Sep–Nov) may be incomplete at export time; the notebook retains the available months and reports eligibility. Treat those estimates as provisional.
- **Spatial leakage:** coordinate-site grouping handles exact/nearly repeated locations but not nearby spatial autocorrelation or sampling clusters. Spatial blocks and leave-year-out tests are subsequent validation steps.
- **Harmonics remain unresolved:** this notebook intentionally tests *only seasonal medians*. A fair comparison with the harmonic models must later use identical field plot-years, site groups, and holdout folds. Do not interpret this notebook's accuracy difference from a previously reported harmonic model as a representation effect until samples and splits are matched.
